In [1]:
!pip install open3d

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 71.5 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt

In [2]:
def generate_voxel(shape_type="cube", size=32):
    voxel = np.zeros((size, size, size))

    if shape_type == "cube":
        voxel[8:24, 8:24, 8:24] = 1

    elif shape_type == "sphere":
        center = size // 2
        for x in range(size):
            for y in range(size):
                for z in range(size):
                    if (x-center)**2 + (y-center)**2 + (z-center)**2 < 100:
                        voxel[x,y,z] = 1

    return voxel

def generate_images(voxel, views=3):
    images = []
    for v in range(views):
        img = voxel.sum(axis=v)
        img = img / (img.max() + 1e-8)
        img = np.stack([img, img, img], axis=0)
        images.append(img)
    return np.array(images)

def get_sample():
    shape = np.random.choice(["cube", "sphere"])
    voxel = generate_voxel(shape)
    images = generate_images(voxel)

    return torch.tensor(images, dtype=torch.float32), \
           torch.tensor(voxel, dtype=torch.float32).unsqueeze(0)

In [5]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc = nn.Linear(64*8*8, 256)

    def forward(self, x):
        x = self.conv(x)
        x = F.adaptive_avg_pool2d(x, (8,8))
        x = x.view(x.size(0), -1)
        return self.fc(x)

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(256, 128*4*4*4)

        self.deconv = nn.Sequential(
            nn.ConvTranspose3d(128, 64, 4, stride=2, padding=1),  # 4 → 8
            nn.ReLU(),

            nn.ConvTranspose3d(64, 32, 4, stride=2, padding=1),   # 8 → 16
            nn.ReLU(),

            nn.ConvTranspose3d(32, 1, 4, stride=2, padding=1),    # 16 → 32
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.fc(x)
        x = x.view(-1, 128, 4, 4, 4)
        return self.deconv(x)

class MultiView3D(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, images):
        B, V, C, H, W = images.shape

        feats = []
        for v in range(V):
            feats.append(self.encoder(images[:, v]))

        feats = torch.stack(feats, dim=1)

        # Fusion (simple)
        fused = feats.mean(dim=1)

        return self.decoder(fused)

In [6]:
model = MultiView3D()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    images, target = get_sample()
    images = images.unsqueeze(0)
    target = target.unsqueeze(0)

    output = model(images)
    loss = F.binary_cross_entropy(output, target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 0.7378
Epoch 1, Loss: 0.7339
Epoch 2, Loss: 0.7207
Epoch 3, Loss: 0.6882
Epoch 4, Loss: 0.5569
Epoch 5, Loss: 0.4267
Epoch 6, Loss: 0.4499
Epoch 7, Loss: 0.2828
Epoch 8, Loss: 0.2666
Epoch 9, Loss: 0.2029


In [7]:
def voxel_to_pointcloud(voxel, threshold=0.5):
    voxel = voxel.detach().cpu().numpy()[0,0]
    points = np.argwhere(voxel > threshold)

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)

    colors = np.zeros_like(points, dtype=float)
    colors[:, 0] = 1.0  # red color
    pcd.colors = o3d.utility.Vector3dVector(colors)

    return pcd

def visualize(voxel):
    pcd = voxel_to_pointcloud(voxel)
    o3d.visualization.draw_plotly([pcd])  # works in Colab

In [8]:
images, target = get_sample()
images = images.unsqueeze(0)

output = model(images)

print("Ground Truth:")
visualize(target.unsqueeze(0))

print("Prediction:")
visualize(output)

Ground Truth:


Prediction:


In [9]:
!pip install scikit-image

In [10]:
from skimage import measure
import open3d as o3d
import numpy as np

def voxel_to_mesh(voxel, threshold=0.5):
    voxel = voxel.detach().cpu().numpy()[0,0]

    verts, faces, normals, _ = measure.marching_cubes(voxel, level=threshold)

    mesh = o3d.geometry.TriangleMesh()
    mesh.vertices = o3d.utility.Vector3dVector(verts)
    mesh.triangles = o3d.utility.Vector3iVector(faces)

    mesh.compute_vertex_normals()

    return mesh

In [13]:
def visualize_mesh(voxel):
    mesh = voxel_to_mesh(voxel)
    mesh.paint_uniform_color([0.1, 0.7, 0.9])
    o3d.visualization.draw_plotly([mesh])

In [14]:
print("Ground Truth Mesh")
visualize_mesh(target.unsqueeze(0))

print("Predicted Mesh")
visualize_mesh(output)

Ground Truth Mesh


Predicted Mesh
